In [1]:
import sys
import os

dust3r_path = './third_party/dust3r'
sys.path.append(dust3r_path)

sys.path.append("/home/arisrei/research/RPL/ORION/third_party/XMem")

from dust3r.inference import inference, load_model
from dust3r.utils.image import load_images
from dust3r.image_pairs import make_pairs
from dust3r.cloud_opt import global_aligner, GlobalAlignerMode
from dust3r.utils.device import to_numpy

import numpy as np
import open3d as o3d
import plotly.graph_objects as go
import os
import PIL
from PIL import Image
from tqdm import tqdm

import torch
from orion.utils.o3d_utils import *
from orion.utils.misc_utils import *
from orion.algos.xmem_tracker import XMemTracker
from orion.algos.oog import *



Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


[robosuite WARNING] No private macro file found! (macros.py:53)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:54)
[robosuite WARNING] To setup, run: python /data/arisrei/miniconda3/envs/dust3r/lib/python3.10/site-packages/robosuite/scripts/setup_macros.py (macros.py:55)


In [2]:
annotation_dir = "/home/arisrei/research/RPL/ORION/annotations/human_demo/yellow_bottle"
dataset_dir = "/home/arisrei/research/RPL/ORION/datasets/iphones/yellow_bottle"
with open(os.path.join(annotation_dir, "config.json"), "r") as f:
    config_dict = json.load(f)

In [3]:
model_path = os.path.join(dust3r_path, "checkpoints/DUSt3R_ViTLarge_BaseDecoder_512_dpt.pth")
device = 'cuda'
batch_size = 1
schedule = 'linear'
lr = 0.01
niter = 300

model = load_model(model_path, device)

img_indexes = []
for i in range(40,65):
    img_indexes.append(i)
img_list = []

for idx in img_indexes:
    img_list.append(os.path.join(dataset_dir, f"images/image{idx}.png"))

# load_images can take a list of images or a directory
images = load_images(img_list, size=512)
pairs = make_pairs(images, scene_graph='complete', prefilter=None, symmetrize=True)
output = inference(pairs, model, device, batch_size=batch_size)

mode = GlobalAlignerMode.PointCloudOptimizer if len(images) > 2 else GlobalAlignerMode.PairViewer
scene = global_aligner(output, device=device, mode=mode)

if mode == GlobalAlignerMode.PointCloudOptimizer:
    loss = scene.compute_global_alignment(init='mst', niter=niter, schedule=schedule, lr=lr)

scene = scene.clean_pointcloud()
# retrieve useful values from scene:
imgs = scene.imgs
focals = scene.get_focals()
intrinsics = scene.get_intrinsics().cpu().detach().numpy()
poses = scene.get_im_poses().cpu().detach().numpy()
pts3d = scene.get_pts3d()

scene.min_conf_thr = float(scene.conf_trf(torch.tensor(3)))
print("min_conf_thr: ", scene.min_conf_thr)
mask = to_numpy(scene.get_masks())

... loading model from ./third_party/dust3r/checkpoints/DUSt3R_ViTLarge_BaseDecoder_512_dpt.pth
instantiating : AsymmetricCroCo3DStereo(enc_depth=24, dec_depth=12, enc_embed_dim=1024, dec_embed_dim=768, enc_num_heads=16, dec_num_heads=12, pos_embed='RoPE100', patch_embed_cls='PatchEmbedDust3R', img_size=(512, 512), head_type='dpt', output_mode='pts3d', depth_mode=('exp', -inf, inf), conf_mode=('exp', 1, inf), landscape_only=False)
<All keys matched successfully>
>> Loading a list of 25 images
 - adding /home/arisrei/research/RPL/ORION/datasets/iphones/yellow_bottle/images/image40.png with resolution 720x1280 --> 288x512
 - adding /home/arisrei/research/RPL/ORION/datasets/iphones/yellow_bottle/images/image41.png with resolution 720x1280 --> 288x512
 - adding /home/arisrei/research/RPL/ORION/datasets/iphones/yellow_bottle/images/image42.png with resolution 720x1280 --> 288x512
 - adding /home/arisrei/research/RPL/ORION/datasets/iphones/yellow_bottle/images/image43.png with resolution 720

100%|██████████| 600/600 [00:48<00:00, 12.41it/s]


 init edge (2*,1*) score=48.283790588378906
 init edge (2,3*) score=46.80001449584961
 init edge (2,0*) score=41.83262634277344
 init edge (2,4*) score=39.49457931518555
 init edge (2,5*) score=37.05359649658203
 init edge (6*,1) score=33.705230712890625
 init edge (2,10*) score=33.61052703857422
 init edge (12*,3) score=33.58042907714844
 init edge (2,7*) score=32.82996368408203
 init edge (2,9*) score=32.57583236694336
 init edge (2,8*) score=32.53429412841797
 init edge (2,11*) score=32.02288055419922
 init edge (12,13*) score=31.47993278503418
 init edge (7,22*) score=11.469247817993164
 init edge (2,24*) score=10.844683647155762
 init edge (2,21*) score=10.200471878051758
 init edge (23*,1) score=10.103670120239258
 init edge (20*,3) score=9.502874374389648
 init edge (5,19*) score=8.070429801940918
 init edge (13,14*) score=32.315067291259766
 init edge (14,15*) score=28.72365951538086
 init edge (16*,15) score=16.713985443115234
 init edge (16,17*) score=9.908065795898438
 init 

100%|██████████| 300/300 [01:18<00:00,  3.84it/s, lr=3.433e-05 loss=0.00390014] 


min_conf_thr:  1.0986123085021973


In [5]:
pts3d = to_numpy(pts3d)
imgs = to_numpy(imgs)
print(len(pts3d), pts3d[0].shape)
pts = np.concatenate([p[m] for p, m in zip(pts3d, mask)])
col = np.concatenate([p[m] for p, m in zip(imgs, mask)])
print(pts.shape, col.shape)

25 (512, 288, 3)
(1149439, 3) (1149439, 3)


In [ ]:
np.save(os.path.join(annotation_dir, "dust3r/scene_points.npy"), pts)
np.save(os.path.join(annotation_dir, "dust3r/scene_colors.npy"), col)
np.save(os.path.join(annotation_dir, "dust3r/intrinsics.npy"), intrinsics)
np.save(os.path.join(annotation_dir, "dust3r/extrinsics.npy"), poses)

5 (512, 288, 3)
(371626, 3) (371626, 3)


In [6]:
pcd_all = o3d.geometry.PointCloud()
pcd_all.points = o3d.utility.Vector3dVector(pts)
pcd_all.colors = o3d.utility.Vector3dVector(col)

pcd_wrapped = O3DPointCloud()
pcd_wrapped.pcd = pcd_all
pcd_wrapped.max_points = 40000
print(np.array(pcd_wrapped.pcd.points).shape, np.array(pcd_wrapped.pcd.colors).shape)
pcd_wrapped.preprocess()
print(np.array(pcd_wrapped.pcd.points).shape, np.array(pcd_wrapped.pcd.colors).shape)
fig = visualize_o3d_point_cloud(pcd_wrapped)
display(fig)

(1149439, 3) (1149439, 3)
(40000, 3) (40000, 3)
